# LSTM 超参数搜索

在 ETTh1 h96 上做 32 组合网格搜索，选出 val_loss 最优配置。

搜索空间：hidden_size × num_layers × dropout × lr × weight_decay = 2^5 = 32 组合

固定参数：epochs=25, patience=5, batch_size=32, seed=216

In [ ]:
import sys
sys.path.insert(0, '')

from scripts.tune_lstm import (
    build_search_grid, DEFAULT_SEARCH_SPACE, DEFAULT_FIXED,
    run_trial, write_summary, set_seed, count_params
)
from models import LSTMModel, TimeSeriesDataset
from models.trainer import resolve_device
from pathlib import Path
import json
import torch

ROOT = Path('').resolve()

## 1. 查看搜索空间

In [ ]:
grid = build_search_grid(DEFAULT_SEARCH_SPACE)
print(f'搜索空间大小: {len(grid)} 组合')
print(f'前3组:')
for i, combo in enumerate(grid[:3]):
    print(f'  {i+1}. {combo}')

## 2. Dry-run：验证参数量范围

In [ ]:
data_dir = ROOT / 'data' / 'processed'
ds = TimeSeriesDataset(data_dir, 'ETTh1', 96, 'train')
input_size = ds.input_size
print(f'ETTh1 h96: input_size={input_size}, target_idx={ds.target_idx}')

param_counts = []
for combo in grid:
    model = LSTMModel(
        input_size=input_size, horizon=96,
        hidden_size=combo['hidden_size'],
        num_layers=combo['num_layers'],
        dropout=combo['dropout']
    )
    params = count_params(model)
    param_counts.append((combo, params))

params_only = [p for _, p in param_counts]
print(f'参数量范围: {min(params_only):,} ~ {max(params_only):,}')
print(f'中位数: {sorted(params_only)[len(params_only)//2]:,}')

min_combo = min(param_counts, key=lambda x: x[1])
max_combo = max(param_counts, key=lambda x: x[1])
print(f'\n最小: {min_combo[1]:,} params  {min_combo[0]}')
print(f'最大: {max_combo[1]:,} params  {max_combo[0]}')

## 3. 运行搜索

可选方式：
- 运行全部 32 组合
- 先用 `--max-trials 5` 快速验证
- 或使用下方单元格逐批运行

In [ ]:
# 方式1：直接运行脚本（推荐在终端执行）
# !python ../scripts/tune_lstm.py --config ../configs/lstm_search.json

# 方式2：在此 notebook 中运行少量 trial 做验证
output_dir = ROOT / 'test_results' / 'h96' / 'ETTh1' / 'lstm'
output_dir.mkdir(parents=True, exist_ok=True)

fixed = {**DEFAULT_FIXED}
data_dir_path = ROOT / 'data' / 'processed'

print(f'Device: {resolve_device(fixed["device"])}')
print(f'Output: {output_dir}')
print(f'Total grid: {len(grid)} configs')

## 4. 分析结果

搜索完成后，读取 summary 并分析。

In [ ]:
import pandas as pd

csv_path = ROOT / 'test_results' / 'h96' / 'ETTh1' / 'lstm' / 'lstm_search_summary.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df = df.sort_values('best_val_loss')
    print(f'共 {len(df)} 组实验')
    print(f'\nTop-5 by val_loss:')
    cols = ['run_name', 'hidden_size', 'num_layers', 'dropout',
            'lr', 'weight_decay', 'model_params', 'best_val_loss', 'MSE', 'R2']
    display(df[cols].head())
else:
    print('搜索结果尚未生成，请先运行搜索。')